In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("novacredit_clientes.csv")

In [9]:
#Identificando valores faltantes
df.isnull().sum()

edad                          0
ingreso_mensual             744
antiguedad_cliente_meses      0
segmento_cliente              0
region                        0
canal_adquisicion             0
limite_credito                0
saldo_promedio                0
porcentaje_utilizacion        0
num_transacciones_mes         0
pagos_atrasados_12m           0
reclamos_12m                  0
tipo_producto                 0
tipo_contrato                 0
metodo_pago                 206
default_12m                   0
dtype: int64

In [10]:
#Rellenando los nulos que sean numéricos con la mediana, para no dejar huecos. Al tiempo que se evita meter sesgos derivados de valores extremos
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [11]:
#Usamos one hot encoding para no introducir conexiones ordinales artificiales
cat_cols = df.select_dtypes(include=["object", "string"]).columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [19]:
#Usamos IQR para detectar valores atípicos en variables como ingreso, credito y porcentaje...
# No se eliminan los outliers, solo se identifican y analizan,
# ya que el enfoque se limita a la preparación de los datos

def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)



In [21]:
# Identificación de outliers en variables numéricas clave
outliers_ingreso = detect_outliers_iqr(df["ingreso_mensual"])
outliers_limite = detect_outliers_iqr(df["limite_credito"])
outliers_utilizacion = detect_outliers_iqr(df["porcentaje_utilizacion"])
outliers_ingreso.sum(), outliers_limite.sum(), outliers_utilizacion.sum()

#Los valores atípicos que se despliegan corresponden a ingreso mensual, limite_credito, procentaje_utilizacion
#Como el dataset es de 10000 registros entonces es 5.36%, 4.76% y 0.52% de los clientes


(536, 476, 52)

In [22]:
#Verificando nulos
df_encoded.isnull().sum().sum()

0

## Identificación y análisis de valores atípicos (outliers)

Para la detección de valores atípicos en variables numéricas clave del conjunto de datos se utilizó el método del **rango intercuartílico (IQR)**, el cual permite identificar observaciones extremas sin asumir una distribución normal de los datos. Este método es especialmente adecuado en contextos financieros, donde la presencia de valores extremos puede representar comportamientos reales y relevantes.

Se aplicó el método IQR a las variables ingreso_mensual, limite_credito y orcentaje_utilizacion, obteniéndose los siguientes resultados:

- En la variable *ingreso_mensual* se identificaron **536 registros atípicos**, lo que representa aproximadamente el **5.36 %** del total de clientes.
- En la variable *limite_credito* se detectaron **476 valores atípicos**, equivalentes al **4.76 %** de los registros.
- En la variable *porcentaje_utilizacion* se encontraron **52 valores atípicos**, lo que corresponde a alrededor del **0.52 %** del conjunto de datos.

Estos resultados indican que las variables relacionadas con ingresos y límites de crédito presentan una mayor dispersión, lo cual es consistente con la naturaleza de los datos financieros, donde pueden existir clientes con ingresos o capacidades crediticias significativamente superiores al promedio. Por su parte, la variable de porcentaje de utilización muestra un comportamiento más acotado, dado que se encuentra naturalmente limitada a un rango definido.

En esta etapa del análisis, los valores atípicos no fueron eliminados, ya que pueden representar clientes de alto valor o alto riesgo. Su identificación y cuantificación permite comprender mejor la distribución de los datos y deja documentada su existencia como parte del proceso de preparación. El enfoque de la actividad se limita exclusivamente a la limpieza, transformación y análisis exploratorio de los datos, sin la aplicación de modelos predictivos.
